# PART-1: GPT-2 Sentiment Analysis

| | SST (5-class) | CFIMDB (binary) |
|---|---|---|
| last-linear-layer | 0.462 | 0.861 |
| full-model | 0.513 | 0.976 |

In [ ]:
from pathlib import Path
import sys, os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'classifier.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

## 1. 구현 검증

In [ ]:
from sanity_check import test_gpt2
test_gpt2('gpt2')

In [ ]:
import numpy as np
import torch
from optimizer_test import test_optimizer
from optimizer import AdamW

ref = torch.tensor(np.load('optimizer_test.npy'))
actual = test_optimizer(AdamW)
assert torch.allclose(ref, actual, atol=1e-6, rtol=1e-4)
print('Optimizer test passed!')

## 2. Fine-tuning

In [ ]:
# SST - last-linear-layer
import importlib
import classifier
from types import SimpleNamespace

importlib.reload(classifier)
classifier.seed_everything(11711)

config = SimpleNamespace(
    filepath='sst-last-linear-classifier.pt',
    lr=1e-3, use_gpu=True, epochs=10, batch_size=64,
    hidden_dropout_prob=0.3,
    train='data/ids-sst-train.csv',
    dev='data/ids-sst-dev.csv',
    test='data/ids-sst-test-student.csv',
    fine_tune_mode='last-linear-layer',
    dev_out='predictions/last-linear-layer-sst-dev-out.csv',
    test_out='predictions/last-linear-layer-sst-test-out.csv',
)
classifier.train(config)
classifier.test(config)

In [ ]:
# SST - full-model
importlib.reload(classifier)
classifier.seed_everything(11711)

config = SimpleNamespace(
    filepath='sst-full-model-classifier.pt',
    lr=1e-5, use_gpu=True, epochs=10, batch_size=64,
    hidden_dropout_prob=0.3,
    train='data/ids-sst-train.csv',
    dev='data/ids-sst-dev.csv',
    test='data/ids-sst-test-student.csv',
    fine_tune_mode='full-model',
    dev_out='predictions/full-model-sst-dev-out.csv',
    test_out='predictions/full-model-sst-test-out.csv',
)
classifier.train(config)
classifier.test(config)

In [ ]:
# CFIMDB - last-linear-layer
importlib.reload(classifier)
classifier.seed_everything(11711)

config = SimpleNamespace(
    filepath='cfimdb-last-linear-classifier.pt',
    lr=1e-3, use_gpu=True, epochs=10, batch_size=8,
    hidden_dropout_prob=0.3,
    train='data/ids-cfimdb-train.csv',
    dev='data/ids-cfimdb-dev.csv',
    test='data/ids-cfimdb-test-student.csv',
    fine_tune_mode='last-linear-layer',
    dev_out='predictions/last-linear-layer-cfimdb-dev-out.csv',
    test_out='predictions/last-linear-layer-cfimdb-test-out.csv',
)
classifier.train(config)
classifier.test(config)

In [ ]:
# CFIMDB - full-model
importlib.reload(classifier)
classifier.seed_everything(11711)

config = SimpleNamespace(
    filepath='cfimdb-full-model-classifier.pt',
    lr=1e-5, use_gpu=True, epochs=10, batch_size=8,
    hidden_dropout_prob=0.3,
    train='data/ids-cfimdb-train.csv',
    dev='data/ids-cfimdb-dev.csv',
    test='data/ids-cfimdb-test-student.csv',
    fine_tune_mode='full-model',
    dev_out='predictions/full-model-cfimdb-dev-out.csv',
    test_out='predictions/full-model-cfimdb-test-out.csv',
)
classifier.train(config)
classifier.test(config)

## 3. 결과 요약

In [ ]:
import csv
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support

EVAL_CONFIGS = [
    ('SST  / last-linear', 'data/ids-sst-dev.csv',     'predictions/last-linear-layer-sst-dev-out.csv'),
    ('SST  / full-model',  'data/ids-sst-dev.csv',     'predictions/full-model-sst-dev-out.csv'),
    ('CFIMDB / last-linear', 'data/ids-cfimdb-dev.csv', 'predictions/last-linear-layer-cfimdb-dev-out.csv'),
    ('CFIMDB / full-model',  'data/ids-cfimdb-dev.csv', 'predictions/full-model-cfimdb-dev-out.csv'),
]

def read_dev(path):
    with open(path, newline='') as f:
        return {r['id'].strip(): int(r['sentiment']) for r in csv.DictReader(f, delimiter='\t')}

def read_pred(path):
    with open(path, newline='') as f:
        return {r['id'].strip(): int(r['Predicted_Sentiment']) for r in csv.DictReader(f)}

results = []
print(f"{'Config':<25} {'Accuracy':>10} {'Macro-F1':>10}")
print('-' * 48)
for label, dev_path, pred_path in EVAL_CONFIGS:
    if not Path(pred_path).exists():
        print(f"{label:<25} {'(no file)':>10}")
        continue
    true = read_dev(dev_path)
    pred = read_pred(pred_path)
    ids  = [k for k in pred if k in true]
    y_true = [true[k] for k in ids]
    y_pred = [pred[k]  for k in ids]
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    results.append({'label': label, 'acc': acc, 'f1': f1, 'y_true': y_true, 'y_pred': y_pred})
    print(f"{label:<25} {acc:>10.4f} {f1:>10.4f}")

## 4. 시각화 & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

if not results:
    print('결과 없음 — Section 3 셀을 먼저 실행하세요.')
else:
    labels = [r['label'] for r in results]
    accs   = [r['acc']   for r in results]
    f1s    = [r['f1']    for r in results]
    x = np.arange(len(labels))
    w = 0.35

    # ── Accuracy & Macro-F1 bar chart ────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#2563eb')
    b2 = ax.bar(x + w/2, f1s,  w, label='Macro-F1', color='#16a34a')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=12, ha='right')
    ax.set_ylim(0, 1.15)
    ax.set_title('Dev Accuracy & Macro-F1')
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    for bar, val in zip(list(b1) + list(b2), accs + f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Confusion matrices ────────────────────────────────────
    n   = len(results)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle('Confusion Matrices (Dev Split)', fontsize=13, fontweight='bold')
    for ax, r in zip(axes, results):
        cm     = confusion_matrix(r['y_true'], r['y_pred'])
        im     = ax.imshow(cm, cmap='Blues')
        thresh = cm.max() / 2
        ax.set_title(r['label'], fontsize=9)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, cm[i, j], ha='center', va='center',
                        color='white' if cm[i, j] > thresh else 'black', fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()